# Brain Tumor MRI — notebook completo para Google Colab

Classificação de imagens de ressonância magnética em `glioma`, `meningioma`, `notumor` ou `pituitary`,
com ResNet-18 pré-treinada no ImageNet e Grad-CAM. O fluxo inclui download, auditoria dos dados,
validação, treinamento, teste reservado, recarga do modelo e exportação dos resultados.
O escopo é classificação de imagens 2D; não inclui segmentação do tumor.

**Uso educacional e de pesquisa. Este modelo não foi validado para diagnóstico clínico.**

## Como executar

1. Abra [Google Colab](https://colab.research.google.com/) e escolha **Arquivo → Fazer upload de notebook** para enviar este arquivo.
2. Selecione **Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU**, se disponível.
   Os nomes dos menus podem aparecer em inglês: **Runtime → Change runtime type**. A disponibilidade de GPU varia.
3. Execute as células em ordem. A primeira execução precisa de internet para instalar pacotes, baixar os dados e obter os pesos ImageNet.
4. Se já tiver o dataset, preencha `DATA_DIR` com a pasta que contém `Training/` e `Testing/`.
5. Ao terminar, execute a célula de exportação e baixe o ZIP. Os arquivos da máquina virtual do Colab são temporários.

O notebook também funciona em Jupyter com Python 3.11/3.12. Sem CUDA, usa CPU e o treinamento demora mais.
Cada execução cria uma pasta própria com modelo, classes, divisão dos dados, métricas e visualização.
Não são necessários arquivos adicionais de código.

**Dados:** [Brain Tumor MRI Dataset — Masoud Nickparvar](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset).
Fixamos a [versão 2](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset/versions/2).
A cópia local conferida tem 7.200 arquivos, com 1.400 por classe em `Training` e 400 em `Testing`,
enquanto a descrição pública informa 7.023 imagens. Por isso, as quantidades abaixo são calculadas após a leitura,
e as métricas usam o número efetivo de imagens após a remoção de duplicatas.

**Correções em relação ao tutorial curto:** o extrator e suas estatísticas de BatchNorm ficam congelados;
a validação escolhe a época; a perda é a média de todos os lotes; duplicatas exatas são auditadas;
o teste é usado somente ao final; Grad-CAM não descongela os pesos nem comprova a presença de uma lesão.

Referência do ambiente: [perguntas frequentes do Colab](https://research.google.com/colaboratory/faq.html).


## 1. Dependências

Execute a instalação antes dos imports. Se o ambiente solicitar reinicialização do kernel, reinicie e prossiga pelos imports.
As versões efetivamente usadas ficam registradas nos resultados. Em caso de falta de memória, reduza `BATCH_SIZE`.
O Grad-CAM usa operações do próprio PyTorch, sem uma dependência adicional.


In [ ]:
%pip install -q "torch>=2.2,<3" "torchvision>=0.17,<1" "kagglehub>=0.3,<1" "numpy>=1.26,<3" "pandas>=2.2,<4" "scikit-learn>=1.4,<2" "matplotlib>=3.8,<4" "Pillow>=10,<13"

In [ ]:
import hashlib
import importlib.metadata
import json
import random
import shutil
from datetime import datetime, timezone
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18
from IPython.display import display
from PIL import Image, ImageOps, UnidentifiedImageError
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import train_test_split

SEED = 42
DATASET_HANDLE = "masoudnickparvar/brain-tumor-mri-dataset/versions/2"
DATA_DIR = None  # Exemplo: Path("/content/brain-tumor-mri-dataset")
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 5
VALIDATION_FRACTION = 0.2
LEARNING_RATE = 1e-3

OUTPUT_DIR = Path("brain_tumor_results") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "TorchVision:", torchvision.__version__)
print("Dispositivo:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA indisponível. No Colab, selecione GPU antes de iniciar o treinamento.")
print("Resultados:", OUTPUT_DIR.resolve())

## 2. Dataset e verificação das imagens

O download usa o cache do KaggleHub. Se houver erro de autenticação, configure o acesso pelo procedimento oficial
ou faça o download manual e defina `DATA_DIR`. Nunca coloque tokens neste notebook.

A leitura abaixo exige as quatro pastas de classes em cada conjunto, verifica os arquivos e calcula SHA-256 dos pixels RGB.
Imagens ilegíveis interrompem a execução para que o problema seja corrigido; não são descartadas silenciosamente.

In [ ]:
if DATA_DIR is None:
    DATA_DIR = Path(kagglehub.dataset_download(DATASET_HANDLE))
else:
    DATA_DIR = Path(DATA_DIR).expanduser().resolve()

print("Dataset:", DATA_DIR)
for split in ("Training", "Testing"):
    if not (DATA_DIR / split).is_dir():
        raise FileNotFoundError(f"Pasta ausente: {DATA_DIR / split}. Ajuste DATA_DIR.")

In [ ]:
def index_images(root):
    rows = []
    for split in ("Training", "Testing"):
        split_dir = Path(root) / split
        found = {p.name for p in split_dir.iterdir() if p.is_dir() and not p.name.startswith(".")}
        if found != set(CLASS_NAMES):
            raise ValueError(f"Classes inesperadas em {split}: {sorted(found)}")
        for label, class_name in enumerate(CLASS_NAMES):
            paths = sorted(p for p in (split_dir / class_name).rglob("*")
                           if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
            if not paths:
                raise ValueError(f"Nenhuma imagem em {split}/{class_name}")
            for path in paths:
                try:
                    with Image.open(path) as image:
                        rgb = image.convert("RGB")
                        rgb.load()
                        digest = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
                        width, height = rgb.size
                except (OSError, UnidentifiedImageError, Image.DecompressionBombError) as exc:
                    raise ValueError(f"Imagem inválida: {path}") from exc
                rows.append({"path": str(path), "original_split": split,
                             "class_name": class_name, "label": label,
                             "hash": digest, "width": width, "height": height})
    return pd.DataFrame(rows)

raw_df = index_images(DATA_DIR)
print(f"Total original: {len(raw_df):,} imagens")
display(pd.crosstab(raw_df["class_name"], raw_df["original_split"]))

## 3. Duplicatas e divisão de validação

Política definida antes do treinamento:

- Excluir todos os pixels idênticos que possuam rótulos conflitantes, registrando os arquivos excluídos.
- Manter somente uma ocorrência de cada imagem idêntica. Quando aparece em treino e teste, preservar a ocorrência de teste.
- Separar 20% do conjunto `Training` restante para validação estratificada, com semente fixa.
- Usar `Testing` somente na avaliação final. A auditoria de integridade não orienta a escolha do modelo.

As métricas se referem ao teste após essa limpeza; não são diretamente comparáveis com resultados sobre o teste original.
O hash encontra pixels exatamente iguais. Ele não identifica versões recomprimidas, recortes ou imagens distintas do mesmo paciente.
Sem identificadores de pacientes, esta divisão não comprova generalização para pacientes novos.

In [ ]:
def clean_and_split(frame):
    conflict_hashes = frame.groupby("hash")["label"].nunique()
    conflict_hashes = set(conflict_hashes[conflict_hashes > 1].index)
    conflicting = frame[frame["hash"].isin(conflict_hashes)].copy()
    eligible = frame[~frame["hash"].isin(conflict_hashes)].copy()
    # A ordem explícita preserva o teste quando uma imagem também aparece no treino.
    eligible["priority"] = eligible["original_split"].map({"Testing": 0, "Training": 1})
    ordered = eligible.sort_values(["priority", "path"])
    duplicates = ordered[ordered.duplicated("hash")].drop(columns="priority")
    unique = ordered.drop_duplicates("hash").drop(columns="priority")
    development = unique[unique["original_split"] == "Training"]
    test = unique[unique["original_split"] == "Testing"].copy()
    if set(development["label"]) != set(range(len(CLASS_NAMES))):
        raise ValueError("Alguma classe ficou sem imagens de treino após a limpeza.")
    if set(test["label"]) != set(range(len(CLASS_NAMES))):
        raise ValueError("Alguma classe ficou sem imagens de teste após a limpeza.")
    if development.groupby("label").size().min() < 2:
        raise ValueError("Cada classe precisa de pelo menos duas imagens de treino distintas.")
    train, validation = train_test_split(
        development, test_size=VALIDATION_FRACTION,
        stratify=development["label"], random_state=SEED,
    )
    return (train.reset_index(drop=True), validation.reset_index(drop=True),
            test.reset_index(drop=True), conflicting, duplicates)

train_df, val_df, test_df, conflicts_df, duplicates_df = clean_and_split(raw_df)
print("Sobreposição original de pixels entre treino e teste:", len(
    set(raw_df.loc[raw_df["original_split"] == "Training", "hash"])
    & set(raw_df.loc[raw_df["original_split"] == "Testing", "hash"])
))
print(f"Excluídas por conflito: {len(conflicts_df)}; duplicatas removidas: {len(duplicates_df)}")
conflicts_df.to_csv(OUTPUT_DIR / "excluded_conflicts.csv", index=False)
duplicates_df.to_csv(OUTPUT_DIR / "excluded_duplicates.csv", index=False)

for left, right in ((train_df, val_df), (train_df, test_df), (val_df, test_df)):
    assert set(left["hash"]).isdisjoint(right["hash"]), "Vazamento de imagens idênticas."
for frame in (train_df, val_df, test_df):
    assert set(frame["label"]) == set(range(len(CLASS_NAMES)))

manifest = pd.concat([
    train_df.assign(split="train"), val_df.assign(split="validation"), test_df.assign(split="test")
], ignore_index=True)
manifest.to_csv(OUTPUT_DIR / "split_manifest.csv", index=False)
counts = pd.crosstab(manifest["class_name"], manifest["split"])
display(counts)
counts.plot.bar(figsize=(9, 4), rot=0, ylabel="Quantidade de imagens", title="Classes após a limpeza")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(9, 10))
for row, class_name in enumerate(CLASS_NAMES):
    candidates = train_df[train_df["class_name"] == class_name]
    samples = candidates.sample(n=min(3, len(candidates)), random_state=SEED)
    for ax in axes[row]:
        ax.axis("off")
    for ax, path in zip(axes[row], samples["path"]):
        with Image.open(path) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(class_name)
plt.suptitle("Exemplos do conjunto de treino")
plt.tight_layout()
plt.show()

## 4. Pipeline de imagens

As imagens são convertidas para RGB e redimensionadas para 224 × 224 com preenchimento preto, preservando a proporção e o campo de visão.
Usamos a normalização dos pesos `ResNet18_Weights.IMAGENET1K_V1`: média `[0.485, 0.456, 0.406]` e desvio padrão `[0.229, 0.224, 0.225]` após a conversão para `[0, 1]`.
O tamanho de redimensionamento do transform é ajustado para 224, pois a imagem já foi preenchida; assim, evitamos o recorte central padrão.
Rotação e zoom leves são aplicados somente no treino. Validação e teste usam o mesmo processamento determinístico da inferência.

In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms(resize_size=IMAGE_SIZE[0], crop_size=IMAGE_SIZE[0])
train_transform = transforms.Compose([
    transforms.RandomRotation(10, fill=0),
    transforms.RandomAffine(degrees=0, scale=(0.92, 1.08), fill=0),
    preprocess,
])


def load_image(path, transform=preprocess, image_size=IMAGE_SIZE):
    with Image.open(path) as image:
        image = ImageOps.pad(image.convert("RGB"), tuple(image_size),
                             method=Image.Resampling.BILINEAR, color=0)
        return transform(image)


class MRIDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["path"].tolist()
        self.labels = frame["label"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        return load_image(self.paths[index], self.transform), self.labels[index]


def make_dataset(frame, training=False):
    dataset = MRIDataset(frame, train_transform if training else preprocess)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=training, num_workers=0,
                      generator=torch.Generator().manual_seed(SEED))

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

images, labels = next(iter(train_ds))
assert tuple(images.shape[1:]) == (3, *IMAGE_SIZE)
assert bool(torch.isfinite(images).all())
assert labels.dtype == torch.int64
assert int(labels.min()) >= 0 and int(labels.max()) < len(CLASS_NAMES)
print("Lote:", images.shape, "Rótulos:", labels.shape)

## 5. Transfer learning com ResNet-18

A ResNet-18 usa pesos pré-treinados no ImageNet-1K, que contém mais de um milhão de imagens de treinamento.
Reutilizamos os filtros aprendidos para extrair características como bordas, formas e texturas das ressonâncias.
Isso não significa que o modelo já tenha aprendido a reconhecer tumores.

Congelamos todos os parâmetros originais e substituímos somente `fc` por uma camada linear de 512 entradas e quatro saídas.
Apenas os **2.052 parâmetros** dessa camada são treinados. O extrator permanece em modo de avaliação, inclusive durante o treino,
para que as médias e variâncias de BatchNorm também permaneçam congeladas.

`CrossEntropyLoss` recebe logits; softmax é aplicado somente na avaliação e na inferência.
A validação escolhe a melhor época pela perda. A acurácia será medida após o treinamento.

Referências: [transfer learning com PyTorch](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
e [ResNet-18 no TorchVision](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html).

In [ ]:
model = resnet18(weights=weights)
model.requires_grad_(False)
model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
model = model.to(DEVICE)
model.eval()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)

trainable_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
assert trainable_names == ["fc.weight", "fc.bias"]
assert sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad) == 2052
print("Modelo: ResNet-18; pesos:", weights.name)
print("Parâmetros treináveis: 2.052, somente fc.weight e fc.bias")

In [ ]:
def run_epoch(loader, training):
    # O extrator permanece em eval para não atualizar as estatísticas de BatchNorm.
    model.eval()
    model.fc.train(training)
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        with torch.set_grad_enabled(training):
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if not bool(torch.isfinite(loss)):
                raise RuntimeError("Perda não finita. Verifique os dados e a taxa de aprendizado.")
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += int((logits.argmax(dim=1) == labels).sum().item())
        total += labels.size(0)
    if total == 0:
        raise ValueError("O conjunto de dados está vazio.")
    return total_loss / total, correct / total


checkpoint_path = OUTPUT_DIR / "best_model.pt"
history = []
batchnorm_before = {
    name: buffer.detach().cpu().clone() for name, buffer in model.named_buffers()
}
best_loss, stale_epochs = float("inf"), 0
for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = run_epoch(train_ds, training=True)
    val_loss, val_accuracy = run_epoch(val_ds, training=False)
    if epoch == 1:
        for name, buffer in model.named_buffers():
            torch.testing.assert_close(buffer.detach().cpu(), batchnorm_before[name], rtol=0, atol=0)
    history.append({"loss": train_loss, "accuracy": train_accuracy,
                    "val_loss": val_loss, "val_accuracy": val_accuracy})
    print(f"Época {epoch:02d}: loss={train_loss:.4f}, accuracy={train_accuracy:.4f}, "
          f"val_loss={val_loss:.4f}, val_accuracy={val_accuracy:.4f}")
    if val_loss < best_loss:
        best_loss, stale_epochs = val_loss, 0
        torch.save({name: value.cpu() for name, value in model.state_dict().items()}, checkpoint_path)
    else:
        stale_epochs += 1
        if stale_epochs >= 3:
            print("Treinamento interrompido após três épocas sem melhora na validação.")
            break

del batchnorm_before
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True))
model.eval()
history_df = pd.DataFrame(history, index=np.arange(1, len(history) + 1))
history_df.to_csv(OUTPUT_DIR / "history.csv", index_label="epoch")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
history_df[["loss", "val_loss"]].plot(ax=axes[0], title="Perda", xlabel="Época")
history_df[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Acurácia", xlabel="Época")
plt.tight_layout()
plt.show()

## 6. Avaliação final no teste reservado

Reporte acurácia, acurácia balanceada, F1 macro, precisão e recall por classe.
A matriz de confusão mostra quais classes o modelo confunde. Compare também com um baseline que sempre prevê
somente a classe mais frequente do treino. Evite ajustar parâmetros depois de observar o teste; isso transformaria o teste em validação.
Uma pontuação alta neste teste não comprova generalização para novos pacientes ou outros hospitais.


In [ ]:
y_true = test_df["label"].to_numpy()
model.eval()
with torch.inference_mode():
    scores = torch.cat([
        model(images.to(DEVICE)).softmax(dim=1).cpu() for images, _ in test_ds
    ]).numpy()
assert scores.shape == (len(test_df), len(CLASS_NAMES))
assert np.isfinite(scores).all()
np.testing.assert_allclose(scores.sum(axis=1), 1.0, atol=1e-5)
y_pred = scores.argmax(axis=1)
majority_label = int(train_df["label"].mode().iloc[0])
baseline_pred = np.full_like(y_true, majority_label)
metrics = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    "majority_baseline_accuracy": float(accuracy_score(y_true, baseline_pred)),
    "test_images": len(test_df),
}
display(pd.Series(metrics, name="Resultado"))
report = classification_report(y_true, y_pred, labels=list(range(len(CLASS_NAMES))),
                               target_names=CLASS_NAMES, output_dict=True, zero_division=0)
display(pd.DataFrame(report).T)
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, labels=list(range(len(CLASS_NAMES))), display_labels=CLASS_NAMES,
    cmap="Blues", xticks_rotation=30,
)
plt.title("Matriz de confusão — teste após limpeza")
plt.tight_layout()
plt.show()

(OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
(OUTPUT_DIR / "classification_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
predictions = test_df[["path", "class_name", "label"]].copy()
predictions["predicted_class"] = [CLASS_NAMES[index] for index in y_pred]
for index, name in enumerate(CLASS_NAMES):
    predictions[f"score_{name}"] = scores[:, index]
predictions.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

## 7. Salvar metadados e prever uma imagem

O arquivo `best_model.pt` contém o `state_dict` completo: pesos treinados, pesos congelados e estatísticas de BatchNorm.
O arquivo `metadata.json` registra a arquitetura, as classes, o preenchimento, a normalização, os parâmetros de treino e as versões das bibliotecas.
Guarde os dois arquivos juntos. A recarga reconstrói a ResNet-18 sem baixar novamente os pesos ImageNet.
A função de inferência usa o mesmo processamento determinístico da validação e do teste.

As saídas softmax são pontuações do modelo; não são probabilidades clínicas calibradas.
A função aceita imagens RGB ou em tons de cinza em JPEG/PNG. Não processa arquivos DICOM ou volumes 3D.
Para testar uma imagem própria no Colab, envie um JPEG/PNG pelo painel **Arquivos** e copie seu caminho para `IMAGE_PATH`.


In [ ]:
metadata = {
    "dataset_handle": DATASET_HANDLE,
    "dataset_path": str(DATA_DIR),
    "class_names": CLASS_NAMES,
    "image_size": list(IMAGE_SIZE),
    "seed": SEED,
    "validation_fraction": VALIDATION_FRACTION,
    "batch_size": BATCH_SIZE,
    "max_epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "best_epoch": int(history_df["val_loss"].idxmin()),
    "architecture": "resnet18",
    "pretrained_weights": weights.name,
    "trainable_layer": "fc",
    "preprocessing": {
        "resize_size": IMAGE_SIZE[0], "crop_size": IMAGE_SIZE[0],
        "mean": list(preprocess.mean), "std": list(preprocess.std),
        "image_mode": "RGB", "geometry": "ImageOps.pad", "fill": 0,
        "interpolation": "bilinear", "antialias": True,
    },
    "split_sizes": {"train": len(train_df), "validation": len(val_df), "test": len(test_df)},
    "excluded_conflicts": len(conflicts_df),
    "excluded_duplicates": len(duplicates_df),
    "versions": {name: importlib.metadata.version(name) for name in (
        "torch", "torchvision", "kagglehub", "numpy", "pandas", "scikit-learn", "matplotlib", "Pillow"
    )},
}
metadata_path = OUTPUT_DIR / "metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

loaded_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
loaded_model = resnet18(weights=None)
loaded_model.fc = nn.Linear(loaded_model.fc.in_features, len(loaded_metadata["class_names"]))
loaded_model.load_state_dict(torch.load(checkpoint_path, map_location="cpu", weights_only=True))
loaded_model = loaded_model.to(DEVICE).eval()


def prepare_image(image_path, model_metadata):
    image_path = Path(image_path).expanduser()
    if not image_path.is_file():
        raise FileNotFoundError(f"Imagem não encontrada: {image_path}")
    if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        raise ValueError("Use uma imagem JPEG ou PNG.")
    config = model_metadata["preprocessing"]
    transform = ResNet18_Weights[model_metadata["pretrained_weights"]].transforms(
        resize_size=config["resize_size"], crop_size=config["crop_size"],
        mean=config["mean"], std=config["std"], antialias=config["antialias"],
    )
    return load_image(image_path, transform, model_metadata["image_size"])


def predict_image(image_path, classifier=loaded_model, model_metadata=loaded_metadata):
    image = prepare_image(image_path, model_metadata)
    classifier.eval()
    device = next(classifier.parameters()).device
    with torch.inference_mode():
        prediction = classifier(image.unsqueeze(0).to(device)).softmax(dim=1).cpu().numpy()[0]
    if prediction.shape != (len(model_metadata["class_names"]),) or not np.isfinite(prediction).all():
        raise ValueError("A saída do modelo é inválida ou incompatível com os metadados.")
    return pd.DataFrame({"classe": model_metadata["class_names"], "pontuacao": prediction}).sort_values(
        "pontuacao", ascending=False, ignore_index=True
    )


# Troque pelo caminho de uma imagem de ressonância magnética para fazer uma previsão.
IMAGE_PATH = test_df.iloc[0]["path"]
display(predict_image(IMAGE_PATH))

# A gravação e a recarga devem preservar as previsões.
probe = load_image(IMAGE_PATH).unsqueeze(0).to(DEVICE)
with torch.inference_mode():
    torch.testing.assert_close(model(probe), loaded_model(probe), rtol=1e-5, atol=1e-6)
print("Modelo e resultados salvos em:", OUTPUT_DIR.resolve())

## 8. Grad-CAM: quais regiões influenciaram a resposta?

Grad-CAM combina as ativações do último bloco convolucional (`layer4[-1]`) com os gradientes do logit da classe escolhida.
O mapa destaca contribuições positivas relativas para essa classe. Por padrão, explicamos a classe prevista; use `TARGET_CLASS`
para inspecionar outra, como `glioma`. Isso não altera a previsão nem treina o modelo.

A entrada exige gradientes para permitir Grad-CAM mesmo com todo o extrator congelado. Os pesos e as estatísticas de BatchNorm permanecem intactos.
A sobreposição usa exatamente a imagem RGB preenchida que o modelo recebeu, evitando desalinhamento com o mapa.
O mapa original tem resolução espacial de 7 × 7, ampliada para 224 × 224; não representa um contorno preciso.

**Como interpretar:** calor sobre bordas, textos ou preenchimento pode indicar dependência de artefatos e merece investigação.
Calor sobre uma lesão é uma pista qualitativa, não prova de acerto ou localização tumoral. A classe `notumor` não implica uma região tumoral esperada.
Não há máscaras de tumor neste fluxo para medir sobreposição. Um mapa sem contribuição positiva permanece vazio, sem inventar uma região quente.
As cores são normalizadas por imagem; sua intensidade não é comparável entre exames e não representa probabilidade de tumor.

Referência: [Selvaraju et al., Grad-CAM](https://arxiv.org/abs/1610.02391).

In [ ]:
def grad_cam(image_path, target_class=None, classifier=loaded_model, model_metadata=loaded_metadata):
    class_names = model_metadata["class_names"]
    if target_class is not None and target_class not in class_names:
        raise ValueError(f"Classe inválida. Use uma destas classes: {class_names}")
    device = next(classifier.parameters()).device
    activations = []
    modes = [(module, module.training) for module in classifier.modules()]
    handle = classifier.layer4[-1].register_forward_hook(
        lambda module, inputs, output: activations.append(output)
    )
    try:
        classifier.eval()
        with torch.inference_mode(False), torch.enable_grad():
            # Gradientes na entrada mantêm o grafo do extrator congelado sem descongelar seus pesos.
            image = prepare_image(image_path, model_metadata)
            batch = image.unsqueeze(0).to(device).requires_grad_(True)
            logits = classifier(batch)
            if logits.shape != (1, len(class_names)) or not bool(torch.isfinite(logits).all()):
                raise ValueError("A saída do modelo é inválida ou incompatível com os metadados.")
            predicted_index = int(logits.argmax(dim=1).item())
            target_index = predicted_index if target_class is None else class_names.index(target_class)
            features = activations[0]
            gradients = torch.autograd.grad(logits[0, target_index], features)[0]
            channel_weights = gradients.mean(dim=(2, 3), keepdim=True)
            heatmap = (channel_weights * features).sum(dim=1, keepdim=True).relu()
            heatmap = nn.functional.interpolate(heatmap, size=batch.shape[-2:], mode="bilinear",
                                                align_corners=False)[0, 0].detach().cpu()
            if not bool(torch.isfinite(heatmap).all()):
                raise ValueError("Grad-CAM produziu valores não finitos.")
            peak = float(heatmap.max())
            if peak > 0:
                heatmap = heatmap / peak
            scores = logits.softmax(dim=1)[0].detach().cpu().numpy()
            config = model_metadata["preprocessing"]
            mean = torch.tensor(config["mean"]).view(3, 1, 1)
            std = torch.tensor(config["std"]).view(3, 1, 1)
            rgb = (image.detach().cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    finally:
        handle.remove()
        for module, training in modes:
            module.training = training
    return {"image": rgb, "heatmap": heatmap.numpy(), "scores": scores,
            "predicted_class": class_names[predicted_index], "target_class": class_names[target_index]}

In [ ]:
# Use o mesmo IMAGE_PATH da previsão ou substitua por outra imagem JPEG/PNG.
TARGET_CLASS = None  # Use None para a classe prevista ou um nome, por exemplo, "glioma".
explanation = grad_cam(IMAGE_PATH, target_class=TARGET_CLASS)
heatmap = explanation["heatmap"]
assert heatmap.shape == explanation["image"].shape[:2]
assert np.isfinite(heatmap).all() and 0 <= heatmap.min() <= heatmap.max() <= 1
if not heatmap.any():
    print("Nenhuma contribuição positiva foi encontrada para esta classe na camada analisada.")

fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
axes[0].imshow(explanation["image"])
axes[0].set_title("Imagem recebida pelo modelo")
heat_artist = axes[1].imshow(heatmap, cmap="inferno", vmin=0, vmax=1)
axes[1].set_title("Grad-CAM")
axes[2].imshow(explanation["image"])
axes[2].imshow(heatmap, cmap="inferno", vmin=0, vmax=1, alpha=0.6 * heatmap)
axes[2].set_title("Sobreposição")
for ax in axes:
    ax.axis("off")
fig.colorbar(heat_artist, ax=axes[1], fraction=0.046, pad=0.04, label="Contribuição positiva relativa")
fig.suptitle(f"Previsão: {explanation['predicted_class']} | Classe explicada: {explanation['target_class']}")
heatmap_path = OUTPUT_DIR / "gradcam.png"
fig.savefig(heatmap_path, dpi=150, bbox_inches="tight")
plt.show()
print("Visualização salva em:", heatmap_path.resolve())

## 9. Exportar o modelo e os resultados

O ZIP reúne `best_model.pt`, `metadata.json`, histórico, divisão dos dados, exclusões,
previsões, métricas e `gradcam.png`. As imagens originais não são copiadas para o ZIP.
No Colab, esta célula inicia o download pelo navegador. Em Jupyter, ela exibe um link local.
Execute a exportação antes de encerrar o ambiente; os arquivos da máquina virtual não são permanentes.


In [ ]:
archive_path = Path(shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR))
assert archive_path.is_file() and archive_path.stat().st_size > 0
print("Resultados exportados:", archive_path.resolve())
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink
    display(FileLink(str(archive_path)))
else:
    files.download(str(archive_path))


## Interpretação e limites

- Este notebook entrega um baseline reproduzível a partir dos arquivos e parâmetros registrados. Hardware e operações paralelas podem causar pequenas diferenças numéricas.
- A auditoria de duplicatas reduz uma fonte de vazamento, mas não substitui uma separação por paciente.
- Origem das imagens, aquisição e anotações visuais podem influenciar as previsões. Avaliação externa é necessária para medir generalização.
- Não interprete a classe `notumor` como garantia de ausência de doença. O modelo só distingue os rótulos aprendidos neste dataset.
- Não há métricas preenchidas de antemão. Elas serão geradas ao executar o treinamento e a avaliação.

### Referências

- [Dataset e autoria — Masoud Nickparvar](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)
- [KaggleHub: download de datasets](https://github.com/Kaggle/kagglehub)
- [PyTorch: transfer learning](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)